# Exercise 09 – Sparse VAR Models for Ecological Time Series
### Workshop Tutorial: From OLS-VAR to Regularized Models

In [ ]:
knitr::opts_chunk$set(
  echo    = TRUE,
  message = FALSE,
  warning = FALSE
)

------------------------------------------------------------------------

# Background

In classical VAR models, the number of parameters grows **quadratically** with the number of variables and lags. For a system with $k$ variables and $p$ lags, we need to estimate $k^2 \cdot p$ autoregressive coefficients — plus constants. This quickly becomes intractable.

**Sparse VAR** models address this by applying *regularization*: a penalty that shrinks small, uninformative coefficients to exactly zero. This results in:

-   **Fewer estimated parameters** → less overfitting
-   **Better out-of-sample forecasts** → more generalizable models
-   **Interpretable interaction networks** → non-zero entries reveal genuine dependencies

In this exercise we use the `bigtime` package, which implements sparse VAR via LASSO (L1) and Hierarchical Lag (HLag) penalties.

------------------------------------------------------------------------

# Exercise 1 — OLS vs. Sparse VAR

## Learning Goals

You will compare classical OLS-based VAR estimation with sparse VAR models on a synthetic dataset where **the true generating process is known**. This lets you directly measure how well each method recovers the true structure.

You will explore how regularization affects:

-   lag selection
-   in-sample and out-of-sample predictive accuracy
-   model complexity (number of estimated parameters)
-   structural recovery of the true coefficient matrix

------------------------------------------------------------------------

## Step 1: Load packages and simulate data

We first generate a synthetic multivariate time series from a known sparse VAR process using `simVAR()`.

### Key function: `simVAR()`

`simVAR()` from the `bigtime` package generates synthetic VAR data with a controllable sparse structure.

| Argument | Description |
|----|----|
| `periods` | Number of time points to generate |
| `k` | Number of variables (dimensions) |
| `p` | True lag order of the generating process |
| `sparsity_pattern` | Type of sparsity. `"lasso"` creates random sparse coefficients |
| `sparsity_options` | A list with extra options; `num_zero` sets how many coefficients are forced to zero |
| `decay` | Controls how quickly coefficients decay over lags (smaller = slower decay) |
| `seed` | Random seed for reproducibility |

The function returns a list. We use `$Y` to extract the time series matrix (T × k).

**Task:** Fill in the blanks to simulate a dataset with 200 time points, 5 variables, and a true lag order of 8.

In [ ]:
library(bigtime)

set.seed(123)

sim_data <- simVAR(
  periods          = ___,          # number of time points
  k                = ___,          # number of variables
  p                = ___,          # true lag order
  sparsity_pattern = "___",        # use "lasso" sparsity
  sparsity_options = list(num_zero = 15),
  seed             = 123456,
  decay            = 0.01
)

# Standardise the time series (zero mean, unit variance per column)
Y <- ___(sim_data$Y)

cat("Dimensions of Y:", nrow(Y), "time points ×", ncol(Y), "variables\n")

> **Why `scale()`?** Standardizing (mean = 0, SD = 1) each variable puts all series on the same scale, preventing variables with large absolute values from dominating the model. This is good practice before fitting VAR models.

------------------------------------------------------------------------

## Step 2: Train/test split

We hold out the last 20% of observations as a **test set** to evaluate out-of-sample forecast performance. Models are only fitted on the training data.

**Task:** Complete the split so that 80% of the data goes into training and 20% into testing.

In [ ]:
# Use 80% of time points for training
T_train <- floor(___ * nrow(Y))

Y_train <- Y[1:T_train, ]
Y_test  <- Y[(T_train + 1):nrow(Y), ]

cat("Training observations:", nrow(Y_train), "\n")
cat("Test observations:    ", nrow(Y_test),  "\n")

------------------------------------------------------------------------

## Step 3: Lag selection for the OLS-VAR

Before fitting a classical VAR, we need to choose the lag order $p$. The `VARselect()` function computes information criteria to guide this choice.

### Key function: `VARselect()`

`VARselect()` from the `vars` package evaluates multiple information criteria for different lag orders.

| Argument | Description |
|----|----|
| `y` | Multivariate time series (matrix or data frame) |
| `lag.max` | Maximum lag order to consider (default: 10) |
| `type` | Deterministic terms: `"const"` (intercept), `"trend"`, `"both"`, or `"none"` |

The function returns a list. `$selection` gives the optimal lag selected by each criterion:

-   **AIC** — Akaike Information Criterion (tends to select more lags)
-   **HQ** — Hannan-Quinn
-   **SC** — Schwarz/BIC (penalizes complexity more heavily)
-   **FPE** — Final Prediction Error

**Task:** Run `VARselect()` on the training data including an intercept term, then extract the AIC-optimal lag.

In [ ]:
library(vars)

# Fit VARselect with an intercept ("const")
lag_selection <- VARselect(Y_train, type = "___")

# Show all criteria
print(lag_selection$selection)

# Extract the AIC-optimal lag order
p_ols <- lag_selection$selection["___"]
cat("\nSelected lag order by AIC:", p_ols, "\n")

> **Note:** AIC often selects a large lag order. In high dimensions, this quickly leads to overparameterization — exactly the problem sparse VAR is designed to solve.

------------------------------------------------------------------------

## Step 4: Fit a sparse VAR model

Now we fit a sparse VAR using `sparseVAR()`. This is the central function of the `bigtime` package.

### Key function: `sparseVAR()`

`sparseVAR()` estimates a regularized VAR model. It applies a penalty to the coefficient matrix during estimation, shrinking irrelevant coefficients to zero.

| Argument | Description |
|----|----|
| `Y` | Multivariate time series matrix (T × k) — **rows are time, columns are variables** |
| `p` | Maximum lag order to include. If omitted, `bigtime` uses a default |
| `VARpen` | Penalty type: `"L1"` for LASSO, `"HLag"` for Hierarchical Lag |
| `selection` | How to choose the regularization strength λ: `"cv"` for cross-validation |

The fitted object contains:

| Field     | Description                                                  |
|-----------|--------------------------------------------------------------|
| `$Phihat` | Estimated coefficient matrix (k × k·p_max)                   |
| `$p`      | Maximum lag order used                                       |
| `$MSFEcv` | Cross-validated mean squared forecast errors across λ values |

**Task:** Fit a sparse VAR with L1 (LASSO) penalty and cross-validation selection.

In [ ]:
fit_sparse_full <- sparseVAR(Y_train,
                             VARpen    = "___",   # use LASSO penalty
                             selection = "___")   # select lambda via cross-validation

cat("Maximum lag order considered:", fit_sparse_full$p, "\n")

------------------------------------------------------------------------

## Step 5: Identify active lags

The coefficient matrix `Phihat` has dimensions k × (k · p_max). Each block of k columns corresponds to one lag. We can check which lag blocks contain at least one non-zero coefficient.

**Task:** Fill in the column index formula and the threshold for "non-zero".

In [ ]:
Phi   <- fit_sparse_full$Phihat
k     <- ncol(Y_train)
p_max <- fit_sparse_full$p

active_lags <- sapply(1:p_max, function(l) {
  # Column indices belonging to lag l
  cols <- ((___ - 1) * k + 1):(l * k)
  # Is any coefficient in this block non-zero?
  any(abs(Phi[, cols]) > ___)
})

cat("Active lag(s) in sparse VAR:", which(active_lags), "\n")
cat("Inactive lag(s):            ", which(!active_lags), "\n")

> **Interpretation:** Lags with all-zero coefficient blocks are completely "switched off" by the regularization. This is the key difference from OLS, which always retains all lags.

------------------------------------------------------------------------

## Step 6: Build a comparison function

We now write a reusable function that fits both an OLS-VAR and a sparse VAR for a given lag order $p$, and returns a comparison of:

1.  In-sample RMSE (training fit quality)
2.  Out-of-sample RMSE (forecast quality on held-out data)
3.  Number of non-zero coefficients (model complexity)

### Key function: `recursiveforecast()`

`recursiveforecast()` from `bigtime` produces multi-step forecasts from a fitted sparse VAR by iterating one-step predictions forward.

| Argument | Description                              |
|----------|------------------------------------------|
| `fit`    | A fitted `sparseVAR` object              |
| `h`      | Forecast horizon (number of steps ahead) |

Returns a list; `$fcst` is the (h × k) matrix of forecasted values.

**Task:** Complete the two lines marked with blanks inside the function.

In [ ]:
compare_models <- function(Y_train, Y_test, p) {

  h <- nrow(Y_test)
  k <- ncol(Y_train)

  # --- Fit both models ---
  fit_ols    <- VAR(Y_train, p = p, type = "const")
  fit_sparse <- sparseVAR(Y_train, p = p, selection = "cv")

  # --- In-sample RMSE ---
  rmse_in_ols    <- sqrt(mean((Y_train[(p + 1):nrow(Y_train), ] - fitted(fit_ols))^2))
  rmse_in_sparse <- sqrt(mean((Y_train[(p + 1):nrow(Y_train), ] - fitted(fit_sparse))^2))

  # --- Out-of-sample forecasts ---
  fc_ols    <- sapply(predict(fit_ols, n.ahead = h)$fcst, function(x) x[, "fcst"])

  # Use recursiveforecast() to forecast h steps ahead with the sparse VAR
  fc_sparse <- as.matrix(recursiveforecast(___, h = ___)$fcst)

  rmse_out_ols    <- sqrt(mean((Y_test - fc_ols)^2))
  rmse_out_sparse <- sqrt(mean((Y_test - fc_sparse)^2))

  # --- Number of non-zero coefficients ---
  coef_ols  <- do.call(rbind, lapply(coef(fit_ols), function(x) x[1:(k * p), 1]))
  nz_ols    <- sum(abs(coef_ols) > 1e-10)
  nz_sparse <- sum(abs(fit_sparse$___) > 1e-10)   # which slot holds the coefficients?

  data.frame(
    p            = p,
    rmse_in_ols  = round(rmse_in_ols,    3),
    rmse_in_sp   = round(rmse_in_sparse,  3),
    rmse_out_ols = round(rmse_out_ols,    3),
    rmse_out_sp  = round(rmse_out_sparse, 3),
    nonzero_ols  = nz_ols,
    nonzero_sp   = nz_sparse
  )
}

------------------------------------------------------------------------

## Step 7: Evaluate across multiple lag orders

**Task:** Run the comparison for lag orders 2, 5, 8, and 15.

In [ ]:
results <- do.call(rbind, lapply(
  c(___, ___, ___, ___),   # fill in the four lag orders
  compare_models,
  Y_train = Y_train,
  Y_test  = Y_test
))

print(results)

> **What to look for:**
>
> -   As $p$ grows, does OLS in-sample RMSE decrease? (Overfitting)\
> -   Does OLS out-of-sample RMSE increase? (Generalization failure)\
> -   Does the sparse VAR maintain stable out-of-sample performance?\
> -   How does `nonzero_ols` vs `nonzero_sp` compare across lags?

------------------------------------------------------------------------

## Step 8: Visualize the comparison

In [ ]:
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))

# Panel 1: In-sample RMSE
plot(results$p, results$rmse_in_ols,
     type = "b", pch = 16, col = "steelblue",
     ylim = range(results$rmse_in_ols, results$rmse_in_sp),
     main = "In-sample RMSE", xlab = "Lag order p", ylab = "RMSE")
lines(results$p, results$rmse_in_sp, type = "b", pch = 16, col = "firebrick")
legend("topleft", legend = c("OLS-VAR", "sparse VAR"),
       col = c("steelblue", "firebrick"), lty = 1, pch = 16, bty = "n")

# Panel 2: Out-of-sample RMSE
plot(results$p, results$rmse_out_ols,
     type = "b", pch = 16, col = "steelblue",
     ylim = range(results$rmse_out_ols, results$rmse_out_sp),
     main = "Out-of-sample RMSE", xlab = "Lag order p", ylab = "RMSE")
lines(results$p, results$rmse_out_sp, type = "b", pch = 16, col = "firebrick")

# Panel 3: Non-zero coefficients
plot(results$p, results$nonzero_ols,
     type = "b", pch = 16, col = "steelblue",
     ylim = range(results$nonzero_ols, results$nonzero_sp),
     main = "Non-zero coefficients", xlab = "Lag order p", ylab = "Count")
lines(results$p, results$nonzero_sp, type = "b", pch = 16, col = "firebrick")

par(mfrow = c(1, 1))

> **Discussion:** The in-sample RMSE of OLS is always lower than sparse VAR — OLS can always fit training data better by using more parameters. But this comes at a cost: the OLS model overfits, and its forecasts deteriorate as $p$ grows. The sparse VAR sacrifices a little in-sample fit in exchange for stable, reliable out-of-sample predictions.

------------------------------------------------------------------------

## Step 9: Coefficient heatmap comparison

Now we compare the estimated coefficient matrices of OLS-VAR and sparse VAR against the **ground truth** at a fixed lag order $p = 8$.

### Fit the models at p = 8

**Task:** Fit both an OLS-VAR and a sparse VAR at `p_comp = 8`.

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)

p_comp    <- ___           # fixed lag order
k         <- ncol(Y_train)
var_names <- paste0("V", 1:k)

# Classical OLS-VAR with intercept
fit_ols_comp <- VAR(Y_train, p = ___, type = "const")

# Sparse VAR with L1 penalty
fit_sp_comp  <- sparseVAR(Y_train, p = ___, selection = "cv", VARpen = "___")

### Extract coefficient matrices

In [ ]:
# OLS: bind per-equation coefficient vectors, drop the intercept row
phi_ols    <- do.call(rbind, lapply(coef(fit_ols_comp), function(x) x[1:(k * p_comp), 1]))

# Sparse VAR: coefficient matrix is stored directly in $Phihat
phi_sparse <- fit_sp_comp$___

# Ground truth from the simulation object
phi_true   <- sim_data$coef_mat[1:k, 1:(k * p_comp)]

### Assign consistent row and column names

In [ ]:
col_nms <- paste0("Lag", rep(1:p_comp, each = k), "_", rep(var_names, times = p_comp))

rownames(phi_ols)    <- var_names
rownames(phi_sparse) <- var_names
rownames(phi_true)   <- var_names
colnames(phi_ols)    <- col_nms
colnames(phi_sparse) <- col_nms
colnames(phi_true)   <- col_nms

### Reshape to long format and plot

In [ ]:
make_long <- function(phi_mat, model_label) {
  as.data.frame(phi_mat) %>%
    tibble::rownames_to_column("Target") %>%
    pivot_longer(cols = -Target,
                 names_to  = c("Lag", "Source"),
                 names_sep = "_",
                 values_to = "Coefficient") %>%
    mutate(Target = factor(Target, levels = rev(var_names)),
           Source = factor(Source, levels = var_names),
           Model  = model_label)
}

phi_long <- bind_rows(
  make_long(phi_true,   "Truth"),
  make_long(phi_ols,    "OLS-VAR"),
  make_long(phi_sparse, "sparse VAR")
)

coef_limit <- max(abs(phi_long$Coefficient))

plot_phi <- function(data, title) {
  ggplot(data, aes(x = Source, y = Target, fill = Coefficient)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = ifelse(Coefficient == 0, "", sprintf("%.2f", Coefficient))),
              size = 2.5, color = "black", fontface = "bold") +
    scale_fill_gradient2(low = "#b2182b", mid = "white", high = "#2166ac",
                         midpoint = 0, limits = c(-coef_limit, coef_limit)) +
    scale_x_discrete(position = "top") +
    facet_wrap(~ Lag, ncol = p_comp) +
    labs(title = title) +
    theme_minimal() +
    theme(axis.text.x.top = element_text(angle = 45, hjust = 0, face = "bold", size = 9),
          axis.text.y     = element_text(face = "bold", size = 10),
          axis.title      = element_blank(),
          panel.grid      = element_blank(),
          strip.text      = element_text(size = 11, face = "bold"),
          plot.title      = element_text(size = 13, face = "bold"))
}

plot_phi(phi_long[phi_long$Model == "Truth",      ], paste0("Ground Truth  (p = ", p_comp, ")"))
plot_phi(phi_long[phi_long$Model == "OLS-VAR",    ], paste0("OLS-VAR  (p = ", p_comp, ")"))
plot_phi(phi_long[phi_long$Model == "sparse VAR", ], paste0("Sparse VAR  (p = ", p_comp, ")"))

### Summarize sparsity

In [ ]:
cat("OLS-VAR    — proportion of zero coefficients:",
    round(mean(abs(phi_ols)    < 1e-10) * 100, 1), "%\n")
cat("Sparse VAR — proportion of zero coefficients:",
    round(mean(abs(phi_sparse) < 1e-10) * 100, 1), "%\n")
cat("True model — proportion of zero coefficients:",
    round(mean(abs(phi_true)   < 1e-10) * 100, 1), "%\n")

------------------------------------------------------------------------

# Exercise 2 — Regularization and Model Selection (Interpretation)

This exercise requires **no new code**. Instead, you will analyze a pre-computed result and develop your understanding of the bias-variance trade-off under regularization.

## Setup

The result shown below was produced by fitting a sparse VAR model with $p = 5$ to a **30-dimensional Lotka–Volterra simulation** across a grid of regularization strengths $\lambda$. For each $\lambda$:

-   **In-sample RMSE** (blue) measures how well the model fits the *training* data
-   **Out-of-sample RMSE** (red) measures forecast accuracy on *held-out* data

The vertical dashed line marks the $\lambda$ that minimizes out-of-sample error.

In [ ]:
library(png)
library(grid)
img <- readPNG("Comparison_fit_lambda.png")
grid.newpage()
grid::grid.raster(img)

------------------------------------------------------------------------

## Discussion Questions

**Part 1 — Role of** $\lambda$

-   What happens to the coefficient matrix as $\lambda$ increases?
-   At $\lambda \to \infty$, what does the model predict?

**Part 2 — In-sample vs. Out-of-sample RMSE**

-   Why does in-sample RMSE *decrease* as $\lambda$ decreases (less regularization)?
-   Why does this not translate to better out-of-sample performance?

**Part 3 — Bias-Variance Trade-off**

-   Which region of the $\lambda$ axis corresponds to **overfitting**?
-   Which region corresponds to **underfitting**?
-   Where is the optimal trade-off?

**Part 4 — Model Selection**

-   Why might the cross-validation optimum differ from the in-sample optimum?
-   What would happen if you used in-sample RMSE to select $\lambda$?

**Part 5 — Regularization Path**

-   How does the number of non-zero coefficients change as $\lambda$ increases?
-   What does a highly sparse model imply about ecological interactions?

------------------------------------------------------------------------

## Key Takeaway

In a 30-species system with $p = 5$ lags, an unregularized VAR would need to estimate $30^2 \times 5 = 4500$ coefficients from only 200 observations. This is completely intractable without regularization. The LASSO penalty provides an automated way to identify the small subset of genuinely important interactions — exactly what is needed for reconstructing ecological interaction networks.

------------------------------------------------------------------------

# Exercise 3 — Sparse VAR for Real Ecological Data

## Learning Goals

You will apply sparse VAR to a **real multivariate time series** from the Baltic Sea system and compare two regularization approaches:

-   **L1 (LASSO)** — penalizes each coefficient independently
-   **HLag (Hierarchical Lag)** — penalizes entire lag blocks, enforcing that if Lag 3 is active, then Lags 1 and 2 must also be active

------------------------------------------------------------------------

## Step 1: Load and prepare the data

In [ ]:
time_series <- as.data.frame(read.csv("time_series_balticsea.csv"))

cat("Dataset dimensions:", nrow(time_series), "time points ×",
    ncol(time_series), "variables\n")

------------------------------------------------------------------------

## Step 2: Fit LASSO and HLag sparse VAR models

**Task:** Fit both models with lag order `p = 3`. Use `"L1"` for LASSO and `"HLag"` for the hierarchical lag model.

| `VARpen` | Description                                     |
|----------|-------------------------------------------------|
| `"L1"`   | LASSO: each coefficient penalized independently |
| `"HLag"` | Hierarchical Lag: enforces lag-wise grouping    |

In [ ]:
Y_baltic <- scale(as.matrix(time_series))

# LASSO sparse VAR — fill in VARpen and selection
fit_lasso <- sparseVAR(Y_baltic, p = 3, VARpen = "___", selection = "___")

# HLag sparse VAR — fill in VARpen
fit_hlag  <- sparseVAR(Y_baltic, p = 3, VARpen = "___", selection = "cv")

cat("Both models fitted successfully.\n")

------------------------------------------------------------------------

## Step 3: Visualize the coefficient matrices

In [ ]:
par(mfrow = c(1, 2))
Matrix::image(Matrix::Matrix(fit_lasso$Phihat), main = "LASSO coefficient matrix")
Matrix::image(Matrix::Matrix(fit_hlag$Phihat),  main = "HLag coefficient matrix")
par(mfrow = c(1, 1))

------------------------------------------------------------------------

## Step 4: Count non-zero coefficients

**Task:** Complete the helper function and fill in the total number of possible coefficients formula ($k^2 \times p$).

In [ ]:
count_nonzero <- function(fit, tol = 1e-6) {
  sum(abs(fit$___) > tol)   # which slot holds the estimated coefficients?
}

cat("Non-zero coefficients:\n")
cat(sprintf("  LASSO : %d\n", count_nonzero(fit_lasso)))
cat(sprintf("  HLag  : %d\n", count_nonzero(fit_hlag)))

k_b <- ncol(Y_baltic)
p_b <- 3
total_possible <- ___ * ___ * ___    # k^2 * p

cat(sprintf("\nTotal possible coefficients: %d\n", total_possible))
cat(sprintf("  LASSO sparsity: %.1f%%\n",
            100 * (1 - count_nonzero(fit_lasso) / total_possible)))
cat(sprintf("  HLag  sparsity: %.1f%%\n",
            100 * (1 - count_nonzero(fit_hlag)  / total_possible)))

------------------------------------------------------------------------

## Step 5: Inspect the lag structure

### Key function: `lagmatrix()`

| Argument     | Description                                   |
|--------------|-----------------------------------------------|
| `fit`        | A fitted `sparseVAR` object                   |
| `returnplot` | If `TRUE`, generates a heatmap of active lags |

**Task:** Call `lagmatrix()` for both models and request a plot.

In [ ]:
L_lasso <- lagmatrix(___, returnplot = ___)
L_hlag  <- lagmatrix(___, returnplot = ___)

------------------------------------------------------------------------

## Step 6: Cross-validation diagnostics

**Task:** Call `plot_cv()` and `diagnostics_plot()` for both models.

In [ ]:
par(mfrow = c(1, 2))
plot_cv(___)
plot_cv(___)
par(mfrow = c(1, 1))

diagnostics_plot(___)
diagnostics_plot(___)

------------------------------------------------------------------------

## Step 7: Compare predictive performance

**Task:** Extract the minimum cross-validated MSFE from both fitted objects and print the better model.

In [ ]:
# Minimum CV-MSFE is stored in $MSFEcv — take the minimum
MSFE_lasso <- ___(fit_lasso$MSFEcv)
MSFE_hlag  <- ___(fit_hlag$MSFEcv)

cat("Minimum CV-MSFE:\n")
cat(sprintf("  LASSO : %.4f\n", MSFE_lasso))
cat(sprintf("  HLag  : %.4f\n", MSFE_hlag))
cat(sprintf("\nBetter model (lower MSFE): %s\n",
            ifelse(MSFE_lasso < MSFE_hlag, "LASSO", "HLag")))

------------------------------------------------------------------------

## Final Discussion

1.  **Model complexity:** Which model is sparser? Does lower complexity always mean better prediction?

2.  **Lag structure:** Does the HLag model produce a more ecologically interpretable lag pattern? Does the constraint that "recent lags matter more" make biological sense for this system?

3.  **Prediction vs. interpretation:** The MSFE comparison tells us which model forecasts better. But for reconstructing an ecological network, is prediction accuracy the most important criterion?

4.  **LASSO vs. HLag — when to use which?**

    -   Use **LASSO** when you have no prior knowledge about lag structure and want maximum flexibility
    -   Use **HLag** when ecological theory suggests that recent time steps should dominate, or when you want a cleaner, more interpretable network

------------------------------------------------------------------------

*End of Exercise 09*